# Model test & validation: Trilinear Grid Interpolation

The 'look-up table interpolating itself' baseline. Key edge cases: (1) exact recovery at training grid nodes (should be exact, since linear interpolation at a known node returns that node's own value), (2) correct linear extrapolation behavior beyond the grid boundary via bounds_error=False, fill_value=None (should NOT return NaN).

In [ ]:
from scipy.interpolate import RegularGridInterpolator

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
warnings.filterwarnings("ignore")

def mape(y_true, y_pred):
    return float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
def r2(y_true, y_pred):
    return float(r2_score(y_true, y_pred))

df_raw = pd.read_csv("../../data/chf_long_clean.csv")
df = df_raw[df_raw.X != 1.0].reset_index(drop=True)
FEATURES = ["P", "G", "X"]
TARGET = "CHF"
sorted_P = sorted(df.P.unique())

# Split A (random, seed 0) -- quick interpolation check
X_all, y_all = df[FEATURES].values, df[TARGET].values
XtrA, XteA, ytrA, yteA = train_test_split(X_all, y_all, test_size=0.2, random_state=0)

# Split C (edge extrapolation) -- the honest test
train_dfC = df[df.P <= 16000].reset_index(drop=True)
test_dfC = df[df.P >= 17000].reset_index(drop=True)
XtrC, ytrC = train_dfC[FEATURES].values, train_dfC[TARGET].values
XteC, yteC = test_dfC[FEATURES].values, test_dfC[TARGET].values

print(f"Split A: {len(XtrA)} train / {len(XteA)} test")
print(f"Split C: {len(XtrC)} train / {len(XteC)} test")


## Fit on Split A (interpolation) and Split C (extrapolation)

In [ ]:

def fit_grid_interpolator(train_df, log_target=False):
    p_u = np.sort(train_df.P.unique()); g_u = np.sort(train_df.G.unique()); x_u = np.sort(train_df.X.unique())
    cube = np.full((len(p_u), len(g_u), len(x_u)), np.nan)
    p_idx = {v: i for i, v in enumerate(p_u)}
    g_idx = {v: i for i, v in enumerate(g_u)}
    x_idx = {v: i for i, v in enumerate(x_u)}
    vals = np.log(train_df.CHF.values) if log_target else train_df.CHF.values
    for (p, g, x), v in zip(train_df[["P", "G", "X"]].values, vals):
        cube[p_idx[p], g_idx[g], x_idx[x]] = v
    interp = RegularGridInterpolator((p_u, g_u, x_u), cube, method="linear", bounds_error=False, fill_value=None)
    return interp

train_dfC_full = train_dfC  # from shared setup
interp_C = fit_grid_interpolator(train_dfC_full, log_target=False)
predC = interp_C(XteC)
print(f"Split C: R2={r2(yteC, predC):.4f}, MAPE={mape(yteC, predC):.2f}%")


## Edge-case tests

(1) Query an interior training grid node exactly -- interpolated value should equal the table value with (near-)zero error. (2) Query at pressures both just beyond and far beyond the training boundary and confirm finite, non-NaN, monotonically-changing (not wildly oscillating) extrapolated values.

In [ ]:

# (1) exact node recovery
sample_row = train_dfC_full.iloc[100]
exact_query = np.array([[sample_row.P, sample_row.G, sample_row.X]])
exact_pred = interp_C(exact_query)[0]
print(f"Exact grid-node query: predicted={exact_pred:.4f}, true={sample_row.CHF:.4f} -- "
      f"{'PASS' if abs(exact_pred - sample_row.CHF) < 1e-6 else 'FAIL'}")

# (2) extrapolation behavior beyond the grid
G_fixed, X_fixed = 2000.0, 0.0
p_probe = np.array([16000, 17000, 21000, 30000, 60000], dtype=float)
probe_pts = np.column_stack([p_probe, np.full_like(p_probe, G_fixed), np.full_like(p_probe, X_fixed)])
probe_pred = interp_C(probe_pts)
print("\nExtrapolated predictions (G=2000, X=0.0), training P maxes out at 16000:")
for p, pred in zip(p_probe, probe_pred):
    print(f"  P={p:>7.0f} kPa -> predicted CHF={pred:.1f}")
print(f"\nNo NaN in extrapolated region: {'PASS' if not np.any(np.isnan(probe_pred)) else 'FAIL'} "
      f"(bounds_error=False, fill_value=None correctly linearly extrapolates rather than "
      f"returning NaN)")
print(f"Extrapolation continues the pressure-decreasing trend (not flat like trees): "
      f"{'PASS' if probe_pred[-1] < probe_pred[0] else 'CHECK -- trend direction unexpected'}")


## Diagnostic plot

In [ ]:

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(p_probe, probe_pred, "o-")
ax.axvline(16000, color="gray", linestyle="--", label="max training P")
ax.set_xlabel("Query pressure (kPa)")
ax.set_ylabel("Predicted CHF")
ax.set_title("Grid interpolator: linear extrapolation continues the trend (unlike trees)")
ax.legend()
plt.tight_layout()
plt.savefig("../results/model_tests_gridinterp.png", dpi=100)
plt.show()
